# Assignment 22 — Embedding Models, Vector Stores & Similarity Search

**Student:** Abhishek Thakare

In this notebook I am testing the main parts of a small retrieval system:
- loading documents
- splitting them into chunks
- creating embeddings
- searching by cosine similarity
- using FAISS and ChromaDB
- comparing Hugging Face, Ollama and OpenAI approaches
- connecting the pieces into one small pipeline

**Important OpenAI limitation:** I currently have no usable OpenAI API credits. I have kept the real OpenAI implementation in the notebook, but I will not make up OpenAI output. If the API call fails because of quota/credits, the notebook records that result and continues with the local experiments.


## Before running

The notebook expects these files:

```text
data/
├── notes.txt
├── data.csv
└── company_overview.pdf
```

For the local Ollama part, `nomic-embed-text` must be installed and Ollama must be running.

For OpenAI, an `OPENAI_API_KEY` is needed. The OpenAI cell below is a real implementation. If the account has no credits, the request may fail with a quota/billing error. That is recorded instead of inventing results.


In [1]:
# Run this only if some packages are missing in the notebook environment.
# %pip install -U langchain langchain-community langchain-text-splitters langchain-huggingface langchain-ollama langchain-chroma sentence-transformers scikit-learn faiss-cpu pypdf pandas chromadb openai langchain-openai


In [2]:
import os
import time
import numpy as np
import pandas as pd

print("Python environment is ready.")
print("OpenAI API key available:", bool(os.getenv("OPENAI_API_KEY")))

Python environment is ready.
OpenAI API key available: True


## Part 1 — Document Loading

In [3]:
from langchain_community.document_loaders import TextLoader, CSVLoader, PyPDFLoader

txt_loader = TextLoader("data/notes.txt")
txt_docs = txt_loader.load()

csv_loader = CSVLoader("data/data.csv")
csv_docs = csv_loader.load()

pdf_loader = PyPDFLoader("data/company_overview.pdf")
pdf_docs = pdf_loader.load()

all_docs = txt_docs + csv_docs + pdf_docs

print("TXT documents:", len(txt_docs))
print("CSV rows/documents:", len(csv_docs))
print("PDF pages:", len(pdf_docs))
print("Total documents:", len(all_docs))

C:\Users\abhis\AppData\Local\Temp\ipykernel_11724\3743267600.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader, CSVLoader, PyPDFLoader


TXT documents: 1
CSV rows/documents: 8
PDF pages: 2
Total documents: 11


In [4]:
for i, doc in enumerate(all_docs[:5], start=1):
    print(f"\nDocument {i}")
    print("-" * 60)
    print("Metadata:", doc.metadata)
    print(doc.page_content[:300])


Document 1
------------------------------------------------------------
Metadata: {'source': 'data/notes.txt'}
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable wa

Document 2
------------------------------------------------------------
Metadata: {'source': 'data/data.csv', 'row': 0}
employee_id: 101
name: Ananya Sharma
department: Engineering
role: Software Developer
years_experience: 3
location: Pune

Document 3
------------------------------------------------------------
Metadata: {'source': 'data/data.csv', 'row': 1}
employee_id: 102
name: Rahul Verma
department: Engineering
role: ML Engineer
years_experience: 5
location: Bengaluru

Document 4
------------------------------------------------------------
Metadata: {'source': 'data/data.cs

### Document loading observation

The three source types are converted into LangChain `Document` objects. The text is now in a common format, which makes the later splitting and embedding steps independent of the original file type.

## Part 2 — Document Splitting / Chunking

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(all_docs)

print("Total chunks:", len(chunks))

for i, chunk in enumerate(chunks[:3], start=1):
    print(f"\nChunk {i}")
    print("-" * 60)
    print("Length:", len(chunk.page_content))
    print("Metadata:", chunk.metadata)
    print(chunk.page_content[:300])

Total chunks: 17

Chunk 1
------------------------------------------------------------
Length: 423
Metadata: {'source': 'data/notes.txt'}
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable wa

Chunk 2
------------------------------------------------------------
Length: 485
Metadata: {'source': 'data/notes.txt'}
Why Document Loaders Matter
Real-world knowledge is scattered across many formats: plain text notes, CSV
spreadsheets, PDF reports, and web pages. Each format stores information
differently, so each one needs its own loading strategy. LangChain solves this
with a family of Document Loader classes th

Chunk 3
------------------------------------------------------------
Length: 149
Metadata: {'source': 'data/notes.txt'}
format. This unifo

### Chunking observation

The documents are split before embedding because retrieval normally works better when each searchable unit is small enough to contain a focused piece of information. The overlap helps prevent important context from being cut exactly at a chunk boundary.

## Part 3 — Task 1: OpenAI Embedding Model


### What I am doing

The required OpenAI model is `text-embedding-3-small`. I first create the embedding object and then send the same document chunks that I use for the other embedding experiments.

I cannot claim an OpenAI vector dimension or timing unless the API call really succeeds.


In [6]:
texts = [chunk.page_content for chunk in chunks]

# Task 1: real OpenAI implementation
# If the account has no usable credits, the request will fail and that result is recorded.

openai_vectors = None
openai_time = None
openai_dimension = None
openai_error = None
openai_embeddings = None

try:
    from langchain_openai import OpenAIEmbeddings

    openai_embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    start = time.perf_counter()
    openai_vectors = openai_embeddings.embed_documents(texts)
    openai_time = time.perf_counter() - start

    if not openai_vectors:
        raise ValueError("OpenAI returned no embedding vectors.")

    openai_dimension = len(openai_vectors[0])

    print("OpenAI embedding call worked.")
    print("Chunks:", len(openai_vectors))
    print("Vector dimension:", openai_dimension)
    print("Time:", round(openai_time, 4), "seconds")
    print("First 10 values:", openai_vectors[0][:10])

except Exception as exc:
    openai_error = exc
    print("OpenAI embedding could not be completed.")
    print("Error type:", type(exc).__name__)
    print("Error:", str(exc))
    print("No OpenAI result is recorded because the API call did not succeed.")


OpenAI embedding could not be completed.
Error type: RateLimitError
Error: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
No OpenAI result is recorded because the API call did not succeed.


### Task 1 observation

On my current account, the OpenAI experiment is expected to fail because there are no usable API credits. The important point is that the OpenAI code is actually implemented; I am only unable to produce genuine output without API access.

If the call succeeds later, the printed dimension and timing should be used in the comparison table below.


## Task 2 — HuggingFace Embedding Model

In [7]:
from sentence_transformers import SentenceTransformer

hf_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

sample_text = chunks[0].page_content
sample_vector = hf_model.encode(sample_text)

print("Sample vector type:", type(sample_vector))
print("Vector dimension:", len(sample_vector))
print("First 20 values:", sample_vector[:20])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Sample vector type: <class 'numpy.ndarray'>
Vector dimension: 384
First 20 values: [-0.07588267  0.0207014  -0.01554933  0.0182001  -0.06249451  0.0406696
  0.11142601  0.0131498  -0.05937485 -0.02930401  0.05081141 -0.00918628
  0.08517379  0.02503171 -0.00049583  0.07913013  0.02954554  0.00697294
 -0.04494064 -0.07320027]


In [8]:
texts = [chunk.page_content for chunk in chunks]

# I time the same list of chunks that I used for the OpenAI experiment.
start = time.perf_counter()
hf_vectors = hf_model.encode(texts)
hf_time = time.perf_counter() - start

if len(hf_vectors) == 0:
    raise ValueError("Hugging Face returned no vectors.")

hf_dimension = len(hf_vectors[0])

print("Hugging Face experiment")
print("-" * 50)
print("Chunks:", len(texts))
print("Vector dimension:", hf_dimension)
print("Time:", round(hf_time, 4), "seconds")
print("Vector shape:", hf_vectors.shape)


Hugging Face experiment
--------------------------------------------------
Chunks: 17
Vector dimension: 384
Time: 0.4105 seconds
Vector shape: (17, 384)


### Important distinction

The embedding dimension tells us the length of the vector. It does **not** by itself tell us which model is better. Retrieval quality, model training, latency, hardware, cost, and the type of text all matter.

## Task 2 — Ollama Embedding Model

In [9]:
from langchain_ollama import OllamaEmbeddings

ollama_embeddings = OllamaEmbeddings(model="nomic-embed-text")

ollama_vectors = None
ollama_time = None
ollama_dimension = None

try:
    sample = ollama_embeddings.embed_query(sample_text)
    print("Ollama sample dimension:", len(sample))
    print("First 20 values:", sample[:20])

    start = time.perf_counter()
    ollama_vectors = ollama_embeddings.embed_documents(texts)
    ollama_time = time.perf_counter() - start

    if not ollama_vectors:
        raise ValueError("Ollama returned no vectors.")

    ollama_dimension = len(ollama_vectors[0])

    print("\nOllama experiment")
    print("-" * 50)
    print("Chunks:", len(texts))
    print("Vector dimension:", ollama_dimension)
    print("Time:", round(ollama_time, 4), "seconds")

except Exception as exc:
    ollama_vectors = None
    ollama_time = None
    ollama_dimension = None
    print("Ollama could not be tested on this run.")
    print("Error:", type(exc).__name__, str(exc))


Ollama sample dimension: 768
First 20 values: [0.007311904, 0.04155906, -0.1383771, -0.0733815, 0.025645304, -0.04950583, -0.023350904, 0.025290847, 0.008319754, 0.01462544, 0.03249248, -0.034567278, 0.06939149, 0.03581936, -0.007227669, -0.010696435, 0.007573357, -0.04080094, -0.03131012, 0.047471374]

Ollama experiment
--------------------------------------------------
Chunks: 17
Vector dimension: 768
Time: 3.8706 seconds


In [10]:
timing_rows = [
    {
        "Model": "HuggingFace all-MiniLM-L6-v2",
        "Chunks": len(texts),
        "Dimension": hf_dimension,
        "Time (seconds)": round(hf_time, 4),
        "Status": "Measured"
    }
]

if ollama_time is not None:
    timing_rows.append({
        "Model": "Ollama nomic-embed-text",
        "Chunks": len(texts),
        "Dimension": ollama_dimension,
        "Time (seconds)": round(ollama_time, 4),
        "Status": "Measured"
    })
else:
    timing_rows.append({
        "Model": "Ollama nomic-embed-text",
        "Chunks": len(texts),
        "Dimension": None,
        "Time (seconds)": None,
        "Status": "Not measured"
    })

if openai_time is not None:
    timing_rows.append({
        "Model": "OpenAI text-embedding-3-small",
        "Chunks": len(texts),
        "Dimension": openai_dimension,
        "Time (seconds)": round(openai_time, 4),
        "Status": "Measured"
    })
else:
    timing_rows.append({
        "Model": "OpenAI text-embedding-3-small",
        "Chunks": len(texts),
        "Dimension": None,
        "Time (seconds)": None,
        "Status": "Not measured - API unavailable"
    })

timing_df = pd.DataFrame(timing_rows)
display(timing_df)


,Model,Chunks,Dimension,Time (seconds),Status
0,HuggingFace all-MiniLM-L6-v2,17,384.0,0.4105,Measured
1,Ollama nomic-embed-text,17,768.0,3.8706,Measured
2,OpenAI text-embedding-3-small,17,NaN,NaN,Not measured - API unavailable


### Task 2 — Performance and ease of use

The table above contains the measurements from this run. The numbers should be interpreted carefully because they depend on my machine, CPU/GPU, model startup time, batching, and network conditions.

- Hugging Face runs locally after the model is downloaded.
- Ollama also runs locally, but it depends on the Ollama service and local hardware.
- OpenAI includes network/API time, so its timing is not a pure model-computation measurement.
- Vector dimension is useful for understanding storage size, but a larger dimension does not automatically mean better retrieval.

For the final written comparison I should use the actual measured values above, not guessed numbers.


## Task 3 — OpenAI vs Hugging Face: When to Prefer Each


### When I would prefer OpenAI

I would choose OpenAI when I want a hosted service and I do not want to download, update and run an embedding model on my own machine. It is also convenient if the rest of the application already uses OpenAI.

The main disadvantages are API dependency, network latency, usage cost and sending the text to an external service.

### When I would prefer Hugging Face

I would choose Hugging Face when I want local inference, more control over the model, or an experiment that does not need a paid embedding API. It is also useful when the data should stay on the local machine.

The disadvantages are local RAM/CPU/GPU usage, model download time and the need to choose and maintain the model myself.


### Cost vs performance — concrete example

The current OpenAI documentation lists `text-embedding-3-small` at **$0.02 per 1 million input tokens**. OpenAI model documentation: https://developers.openai.com/api/docs/models/text-embedding-3-small

That means, ignoring other application costs:

| Input volume | Approx. OpenAI embedding cost |
|---:|---:|
| 100,000 tokens | $0.002 |
| 1,000,000 tokens | $0.02 |
| 10,000,000 tokens | $0.20 |
| 100,000,000 tokens | $2.00 |

These are API charges, not measurements of this assignment.

For Hugging Face there is no OpenAI API charge when the model runs locally, but that does **not** mean the computation is free. The machine uses RAM, CPU/GPU time, storage and electricity. For a small student experiment, that local cost may be practically negligible. For a production system with large traffic, the hardware and maintenance cost can become important.

So the trade-off is not simply "paid vs free". It is more like:

**OpenAI:** pay per usage + simple managed inference  
**Hugging Face:** local compute + more control + no per-request API bill

For performance, I should compare the measured times from Task 2 and also consider whether the model gives useful retrieval results.


## Part 4 — Task 4: Manual Similarity Search

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

def similarity_search(query, top_k=3):
    # Check the inputs first so a bad call gives a clear error.
    if not isinstance(query, str) or query.strip() == "":
        raise ValueError("query must be a non-empty string")

    if isinstance(top_k, bool) or not isinstance(top_k, int):
        raise TypeError("top_k must be an integer")

    if top_k <= 0:
        raise ValueError("top_k must be greater than zero")

    if len(chunks) == 0:
        raise ValueError("There are no document chunks to search")

    if hf_vectors is None or len(hf_vectors) == 0:
        raise ValueError("Document embeddings have not been created")

    if len(hf_vectors) != len(chunks):
        raise ValueError("Number of vectors does not match number of chunks")

    query_vector = hf_model.encode(query)

    if len(query_vector) != len(hf_vectors[0]):
        raise ValueError("Query vector dimension does not match document vectors")

    scores = cosine_similarity([query_vector], hf_vectors)[0]

    k = min(top_k, len(chunks))
    best_indices = np.argsort(scores)[-k:][::-1]

    results = []
    for idx in best_indices:
        results.append({
            "score": float(scores[idx]),
            "document": chunks[idx]
        })

    return results


In [12]:
query = "What is a Personal Knowledge Assistant?"
results = similarity_search(query, top_k=3)

for rank, result in enumerate(results, start=1):
    print(f"\nRank {rank}")
    print("=" * 60)
    print("Score:", round(result["score"], 4))
    print(result["document"].page_content[:500])


Rank 1
Score: 0.6714
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Rank 2
Score: 0.2666
Section 3: Onboarding Checklist
New employees complete training modules covering Python fundamentals, React basics, and cloud
platform tools such as Microsoft Power Platform, including Power Apps, Power Automate, and Logic
Apps.

Rank 3
Score: 0.2035
employee_id: 103
name: Priya Nair
department: Product
role: Product Manager
years_experience: 4
location: Pune


### Task 4 — Validation tests

In [13]:
validation_tests = [
    ("empty query", lambda: similarity_search("", 3)),
    ("wrong top_k type", lambda: similarity_search("hello", "3")),
    ("zero top_k", lambda: similarity_search("hello", 0)),
    ("negative top_k", lambda: similarity_search("hello", -1)),
]

for name, test in validation_tests:
    try:
        test()
        print(name, "FAILED - no error was raised")
    except (ValueError, TypeError) as exc:
        print(name, "PASSED -", type(exc).__name__, str(exc))

large_k_results = similarity_search("hello", 100000)
print("\nLarge top_k test returned:", len(large_k_results), "results")
print("The value was capped at the number of chunks.")


empty query PASSED - ValueError query must be a non-empty string
wrong top_k type PASSED - TypeError top_k must be an integer
zero top_k PASSED - ValueError top_k must be greater than zero
negative top_k PASSED - ValueError top_k must be greater than zero

Large top_k test returned: 17 results
The value was capped at the number of chunks.


### Task 4 observation

The manual search makes the retrieval process visible:

1. embed the query using the same embedding model as the documents;
2. calculate cosine similarity between the query vector and every document vector;
3. sort the scores;
4. return the top results.

I added validation because a retrieval function should fail early when the query, `top_k`, document list, or vector dimensions are invalid.


## Task 5 — Similarity Search with LangChain

In [14]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

hf_embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

faiss_db = FAISS.from_documents(chunks, hf_embeddings)

query = "What is a Personal Knowledge Assistant?"
langchain_results = faiss_db.similarity_search(query, k=3)

for rank, doc in enumerate(langchain_results, start=1):
    print(f"\nRank {rank}")
    print("=" * 60)
    print(doc.page_content[:500])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]


Rank 1
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Rank 2
Section 3: Onboarding Checklist
New employees complete training modules covering Python fundamentals, React basics, and cloud
platform tools such as Microsoft Power Platform, including Power Apps, Power Automate, and Logic
Apps.

Rank 3
employee_id: 103
name: Priya Nair
department: Product
role: Product Manager
years_experience: 4
location: Pune


### Manual search vs LangChain

The manual implementation exposes the individual steps: query embedding, similarity calculation, sorting, and selection.

The LangChain vector-store abstraction hides those implementation details behind `similarity_search`. This makes application code shorter and also lets the same retrieval interface work with different vector stores.

## OpenAI comparison for the similarity-search stage

The assignment asks for OpenAI similarity search. The code path is implemented in Task 1 and can be used to build an OpenAI vector store when embeddings are available.

Because my current account has no usable credits, I cannot truthfully report OpenAI retrieval results. The Hugging Face version is used for the working local similarity-search demonstration.


## Task 6 — FAISS Vector Store

In [15]:
print("FAISS index created:", type(faiss_db).__name__)

faiss_results = faiss_db.similarity_search(
    "What is a Personal Knowledge Assistant?",
    k=3
)

for rank, doc in enumerate(faiss_results, start=1):
    print(f"\nFAISS Result {rank}")
    print("-" * 60)
    print(doc.page_content[:500])

FAISS index created: FAISS

FAISS Result 1
------------------------------------------------------------
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

FAISS Result 2
------------------------------------------------------------
Section 3: Onboarding Checklist
New employees complete training modules covering Python fundamentals, React basics, and cloud
platform tools such as Microsoft Power Platform, including Power Apps, Power Automate, and Logic
Apps.

FAISS Result 3
------------------------------------------------------------
employee_id: 103
name: Priya Nair
department: Product
role: Product Manager
years_experie

## Task 7 — FAISS Persistence

In [16]:
faiss_path = "faiss_index"

faiss_db.save_local(faiss_path)
print("FAISS index saved to:", faiss_path)

loaded_faiss_db = FAISS.load_local(
    faiss_path,
    hf_embeddings,
    allow_dangerous_deserialization=True
)

print("FAISS index reloaded successfully.")

reloaded_results = loaded_faiss_db.similarity_search(
    "What is a Personal Knowledge Assistant?",
    k=3
)

for rank, doc in enumerate(reloaded_results, start=1):
    print(f"\nReloaded FAISS Result {rank}")
    print(doc.page_content[:500])

FAISS index saved to: faiss_index
FAISS index reloaded successfully.

Reloaded FAISS Result 1
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Reloaded FAISS Result 2
Section 3: Onboarding Checklist
New employees complete training modules covering Python fundamentals, React basics, and cloud
platform tools such as Microsoft Power Platform, including Power Apps, Power Automate, and Logic
Apps.

Reloaded FAISS Result 3
employee_id: 103
name: Priya Nair
department: Product
role: Product Manager
years_experience: 4
location: Pune


## Task 8 — ChromaDB Vector Store

In [17]:
try:
    from langchain_chroma import Chroma
except ImportError:
    from langchain_community.vectorstores import Chroma

chroma_db = Chroma.from_documents(
    documents=chunks,
    embedding=hf_embeddings,
    persist_directory="./chroma_db"
)

print("Chroma object:", type(chroma_db).__name__)

chroma_results = chroma_db.similarity_search(
    "What is a Personal Knowledge Assistant?",
    k=3
)

for rank, doc in enumerate(chroma_results, start=1):
    print(f"\nChroma Result {rank}")
    print("-" * 60)
    print(doc.page_content[:500])

Chroma object: Chroma

Chroma Result 1
------------------------------------------------------------
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Chroma Result 2
------------------------------------------------------------
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a langua

## OpenAI comparison limitation

FAISS and Chroma can store embeddings from OpenAI just like embeddings from other providers, but the vectors must first be generated successfully.

Therefore I do not mix Hugging Face/Ollama vectors with OpenAI vectors in the same index. If OpenAI becomes available, I rebuild a separate FAISS and Chroma store using OpenAI embeddings and compare the returned chunks.


In [18]:
reloaded_chroma_db = Chroma(
    persist_directory="./chroma_db",
    embedding_function=hf_embeddings
)

reloaded_chroma_results = reloaded_chroma_db.similarity_search(
    "What is a Personal Knowledge Assistant?",
    k=3
)

print("Chroma reloaded successfully.")

for rank, doc in enumerate(reloaded_chroma_results, start=1):
    print(f"\nReloaded Chroma Result {rank}")
    print(doc.page_content[:500])

Chroma reloaded successfully.

Reloaded Chroma Result 1
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Reloaded Chroma Result 2
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented generation (RAG) system, we first need a reliable way to
ingest data from many different formats and turn it into clean, structured text
that a language model can reason over.

Reloaded Chroma Result 3
Personal Knowledge Assistant - Project Notes

### FAISS vs ChromaDB

FAISS is primarily a similarity-search/indexing library. It is useful when the application needs fast local vector indexing and searching.

ChromaDB provides a more database-like interface for storing embeddings, metadata, and collections with persistence.

Both can perform vector similarity search, but their operational abstractions and persistence features differ.

## Task 9 — FAISS vs ChromaDB Comparison

In [19]:
comparison_df = pd.DataFrame([
    {
        "Feature": "Primary role",
        "FAISS": "Vector similarity search/indexing",
        "ChromaDB": "Vector database / collection storage"
    },
    {
        "Feature": "Persistence",
        "FAISS": "Save/load index explicitly",
        "ChromaDB": "Persistent directory/collection"
    },
    {
        "Feature": "Metadata",
        "FAISS": "Supported through LangChain documents",
        "ChromaDB": "Built into document/collection workflow"
    },
    {
        "Feature": "Good fit",
        "FAISS": "Fast local retrieval/indexing",
        "ChromaDB": "Persistent application-oriented storage"
    },
])

display(comparison_df)

,Feature,FAISS,ChromaDB
0,Primary role,Vector similarity search/indexing,Vector database / collection storage
1,Persistence,Save/load index explicitly,Persistent directory/collection
2,Metadata,Supported through LangChain documents,Built into document/collection workflow
3,Good fit,Fast local retrieval/indexing,Persistent application-oriented storage


## Part 7 — Ollama + Vector Stores

In [20]:
ollama_available = ollama_vectors is not None

if ollama_available:
    from langchain_ollama import OllamaEmbeddings
    ollama_model = OllamaEmbeddings(model="nomic-embed-text")
    print("Ollama embedding model is ready.")
else:
    ollama_model = None
    print("Ollama vector-store tests will be skipped because the embedding experiment was unavailable.")

Ollama embedding model is ready.


### Task 10 — Embedding × Vector Store combinations

In [21]:
combination_results = []

def test_vector_store(name, vector_store, query):
    if vector_store is None:
        return {
            "Combination": name,
            "Status": "Not tested",
            "Top result": ""
        }

    try:
        results = vector_store.similarity_search(query, k=3)

        if not results:
            return {
                "Combination": name,
                "Status": "No results",
                "Top result": ""
            }

        return {
            "Combination": name,
            "Status": "Tested",
            "Top result": results[0].page_content[:180]
        }

    except Exception as exc:
        return {
            "Combination": name,
            "Status": "Error: " + type(exc).__name__,
            "Top result": str(exc)[:180]
        }

query = "What is a Personal Knowledge Assistant?"

# Hugging Face + FAISS
combination_results.append(
    test_vector_store("HuggingFace -> FAISS", faiss_db, query)
)

# Hugging Face + Chroma
combination_results.append(
    test_vector_store("HuggingFace -> Chroma", chroma_db, query)
)

# Ollama combinations
if ollama_model is not None:
    ollama_faiss = FAISS.from_documents(chunks, ollama_model)
    combination_results.append(
        test_vector_store("Ollama -> FAISS", ollama_faiss, query)
    )

    ollama_chroma = Chroma.from_documents(
        documents=chunks,
        embedding=ollama_model,
        collection_name="assignment22_ollama"
    )
    combination_results.append(
        test_vector_store("Ollama -> Chroma", ollama_chroma, query)
    )
else:
    combination_results.append(
        test_vector_store("Ollama -> FAISS", None, query)
    )
    combination_results.append(
        test_vector_store("Ollama -> Chroma", None, query)
    )

# OpenAI combinations only make sense if the OpenAI embedding call succeeded.
if openai_embeddings is not None and openai_vectors is not None:
    openai_faiss = FAISS.from_documents(chunks, openai_embeddings)
    openai_chroma = Chroma.from_documents(
        documents=chunks,
        embedding=openai_embeddings,
        collection_name="assignment22_openai"
    )

    combination_results.append(
        test_vector_store("OpenAI -> FAISS", openai_faiss, query)
    )
    combination_results.append(
        test_vector_store("OpenAI -> Chroma", openai_chroma, query)
    )
else:
    combination_results.append(
        test_vector_store("OpenAI -> FAISS", None, query)
    )
    combination_results.append(
        test_vector_store("OpenAI -> Chroma", None, query)
    )

combination_df = pd.DataFrame(combination_results)
display(combination_df)


,Combination,Status,Top result
0,HuggingFace -> FAISS,Tested,Personal Knowledge Assistant - Project Notes\n...
1,HuggingFace -> Chroma,Tested,Personal Knowledge Assistant - Project Notes\n...
2,Ollama -> FAISS,Tested,Personal Knowledge Assistant - Project Notes\n...
3,Ollama -> Chroma,Tested,Personal Knowledge Assistant - Project Notes\n...
4,OpenAI -> FAISS,Not tested,
5,OpenAI -> Chroma,Not tested,


### Task 10 note about OpenAI combinations

The OpenAI → FAISS and OpenAI → Chroma tests are conditional because a vector store cannot be built from an embedding model that did not return vectors.

With my current zero-credit OpenAI account, those two combinations will be marked **Not tested**. This is a limitation of the environment, not a fabricated result.


## Compare Retrieval Results Across Backends

In [22]:
def show_results(label, vector_store, query):
    print(f"\n{label}")
    print("=" * 70)

    if vector_store is None:
        print("Not tested on this run.")
        return

    try:
        results = vector_store.similarity_search(query, k=3)

        if not results:
            print("No results returned.")
            return

        for rank, doc in enumerate(results, start=1):
            print(f"\nRank {rank}")
            print(doc.page_content[:350].replace("\n", " "))

    except Exception as exc:
        print("Search failed:", type(exc).__name__, str(exc))

show_results("HuggingFace -> FAISS", faiss_db, query)
show_results("HuggingFace -> Chroma", chroma_db, query)

if ollama_model is not None:
    show_results("Ollama -> FAISS", ollama_faiss, query)
    show_results("Ollama -> Chroma", ollama_chroma, query)

if openai_vectors is not None:
    show_results("OpenAI -> FAISS", openai_faiss, query)
    show_results("OpenAI -> Chroma", openai_chroma, query)
else:
    print("\nOpenAI -> FAISS / Chroma were not tested because OpenAI embeddings were unavailable.")



HuggingFace -> FAISS

Rank 1
Personal Knowledge Assistant - Project Notes  Introduction This project aims to build a Personal Knowledge Assistant that can answer questions from a variety of personal and company data sources. Before we can build any kind of retrieval-augmented generation (RAG) system, we first need a reliable way to ingest data from many different formats and t

Rank 2
Section 3: Onboarding Checklist New employees complete training modules covering Python fundamentals, React basics, and cloud platform tools such as Microsoft Power Platform, including Power Apps, Power Automate, and Logic Apps.

Rank 3
employee_id: 103 name: Priya Nair department: Product role: Product Manager years_experience: 4 location: Pune

HuggingFace -> Chroma

Rank 1
Personal Knowledge Assistant - Project Notes  Introduction This project aims to build a Personal Knowledge Assistant that can answer questions from a variety of personal and company data sources. Before we can build any kind of retr

### What to look for

Do not assume that every backend will return identical results.

The vector store is responsible for storing/searching vectors, while the embedding model determines the vector representation. Changing the embedding model can change retrieval rankings even when the source documents and query remain the same.

## Reusable Pipeline

In [23]:
from langchain_community.vectorstores import FAISS

def create_embedding_model(provider):
    # I keep the provider choice here so I can change the embedding
    # without changing the vector-store code below.

    if provider == "huggingface":
        model = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
    elif provider == "ollama":
        model = OllamaEmbeddings(model="nomic-embed-text")
    elif provider == "openai":
        from langchain_openai import OpenAIEmbeddings
        model = OpenAIEmbeddings(model="text-embedding-3-small")
    else:
        raise ValueError("Use huggingface, ollama or openai")

    return model


In [24]:
def create_vector_store(documents, embedding_model, store_type):
    # The embedding model and the vector store do different jobs.
    # This function only decides where the vectors should be stored.

    if store_type == "faiss":
        return FAISS.from_documents(documents, embedding_model)

    if store_type == "chroma":
        return Chroma.from_documents(
            documents=documents,
            embedding=embedding_model,
            collection_name="assignment22_pipeline"
        )

    raise ValueError("Use faiss or chroma")


In [25]:
def build_pipeline(embedding_provider="huggingface", store_type="faiss"):
    model = create_embedding_model(embedding_provider)
    store = create_vector_store(chunks, model, store_type)
    return store


In [26]:
# I use Hugging Face here because it is available locally.
pipeline = build_pipeline(
    embedding_provider="huggingface",
    store_type="faiss"
)

pipeline_query = "What technologies are used for employee onboarding?"
pipeline_results = pipeline.similarity_search(pipeline_query, k=3)

print("Query:", pipeline_query)

for rank, doc in enumerate(pipeline_results, start=1):
    print(f"\nPipeline result {rank}")
    print(doc.page_content[:400])


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Query: What technologies are used for employee onboarding?

Pipeline result 1
Section 3: Onboarding Checklist
New employees complete training modules covering Python fundamentals, React basics, and cloud
platform tools such as Microsoft Power Platform, including Power Apps, Power Automate, and Logic
Apps.

Pipeline result 2
Company Overview Document
This document provides a short overview of our engineering practices and onboarding process for new
employees joining the Software Product Development team. It is intended to be used as sample
content for testing document loaders.
Section 1: Engineering Practices
Our engineering team follows clean code principles, structured requirement gathering through SRS
documents, an

Pipeline result 3
Personal Knowledge Assistant - Project Notes

Introduction
This project aims to build a Personal Knowledge Assistant that can answer questions
from a variety of personal and company data sources. Before we can build any kind
of retrieval-augmented genera

## Task 10 — End-to-End Pipeline

The pipeline I built is:

**Documents → Chunks → Embeddings → Vector Store → Similarity Search**

I tested the pieces separately first and then used the same idea in one pipeline.

The main thing I learned here is that the embedding model and vector store are separate. I can change the embedding model without changing the basic search call, as long as the vector store is built using that model's embeddings.


## Task 11 — Observations & Insights


### 1. Why embeddings matter

A normal keyword search looks for matching words. An embedding search turns the text into vectors and compares the vectors, so two pieces of text can be close even when they use different words.

For example, a question about "forgotten login details" may retrieve a chunk talking about "password reset" because their meanings are related.

### 2. Why a vector store is useful

After the documents are embedded, I need somewhere to keep the vectors and the original document information. FAISS and Chroma both provide similarity search, but they have different storage and application-oriented features.

With a larger collection, using a proper vector index becomes more important because the application needs a practical way to find nearest vectors.

### 3. What the retrieval part contributes to RAG

The retrieval part is the step before generation:

```text
User question
      ↓
Question embedding
      ↓
Similarity search
      ↓
Top relevant chunks
      ↓
LLM gets those chunks as context
      ↓
Answer
```

The LLM is not responsible for searching my local files by itself. The retrieval system first selects useful evidence and then a generation model can use that evidence.

### 4. What I should report from my run

I should fill this section from the actual notebook output:
- measured Hugging Face/Ollama/OpenAI times;
- vector dimensions that were really produced;
- which queries returned useful chunks;
- whether FAISS and Chroma returned the same top results;
- whether Ollama worked on my machine;
- whether OpenAI was unavailable because of credits.

### 5. Important limitation of this submission

I cannot honestly compare OpenAI retrieval quality with the local models until the OpenAI embedding call succeeds. I therefore mark the OpenAI experiment as unavailable instead of inventing a score or result.

This is still useful for understanding the architecture: the same chunks can be embedded by different providers, and each provider's vectors can then be placed into its own FAISS or Chroma collection.


## Final Checklist Before Submission

In [27]:
checklist = {
    "Documents loaded": len(all_docs) > 0,
    "Chunks created": len(chunks) > 0,
    "HuggingFace embeddings executed": hf_vectors is not None,
    "HuggingFace timing measured": hf_time is not None,
    "Ollama embeddings executed": ollama_vectors is not None,
    "OpenAI embeddings executed": openai_vectors is not None,
    "FAISS tested": faiss_db is not None,
    "FAISS persistence tested": loaded_faiss_db is not None,
    "Chroma tested": chroma_db is not None,
    "Chroma persistence tested": reloaded_chroma_db is not None,
    "Similarity validation tested": True,
}

for item, passed in checklist.items():
    print(f"[{'PASS' if passed else 'NOT RUN'}] {item}: {passed}")


[PASS] Documents loaded: True
[PASS] Chunks created: True
[PASS] HuggingFace embeddings executed: True
[PASS] HuggingFace timing measured: True
[PASS] Ollama embeddings executed: True
[NOT RUN] OpenAI embeddings executed: False
[PASS] FAISS tested: True
[PASS] FAISS persistence tested: True
[PASS] Chroma tested: True
[PASS] Chroma persistence tested: True
[PASS] Similarity validation tested: True


## Honest OpenAI note

The OpenAI implementation is present in Task 1 and the later vector-store tests are also present.

On my current account I have zero usable OpenAI credits, so I cannot produce genuine OpenAI vectors, timings or retrieval results. I will not copy numbers from documentation or another run and present them as my own experiment.

If credits become available, I only need to rerun the OpenAI cell and then rerun the OpenAI → FAISS/Chroma combination cells.


## Final reflection

The main lesson from this assignment is that retrieval is a chain of separate steps:

**documents → chunks → embeddings → vector store → similarity search**

The embedding model decides how text is represented as vectors. The vector store keeps those vectors searchable. Similarity search then finds chunks that are close to the user's question.

That separation is what makes the same retrieval idea reusable with different embedding models and different vector stores.
